# 🌿 AgriLens 22-Class Hybrid Model Verification

This notebook verifies the performance of the newly expanded **22-class Hybrid VQC model**. It loads the test dataset and generates a comprehensive classification report.

In [1]:
import sys, os
import torch
sys.path.insert(0, os.path.abspath('../src'))

from utils.config import CFG
from data.loader import load_dataset
from training.evaluate import evaluate_model
from models.vqc import HybridQuantumClassifier
from data.preprocess import CropDiseaseDataset, get_eval_transforms
from torch.utils.data import DataLoader

# Ensure we use the 22-class configuration
print(f"Target Classes: {len(CFG.CLASSES)}")
print(f"Device: {CFG.DEVICE}")

Target Classes: 22
Device: cuda


c:\users\balaj\appdata\local\programs\python\python38\lib\site-packages\pennylane_lightning\lightning_qubit\lightning_qubit.py:822: UserWarning: Pre-compiled binaries for lightning.qubit are not available. Falling back to using the Python-based default.qubit implementation. To manually compile from source, follow the instructions at https://pennylane-lightning.readthedocs.io/en/latest/installation.html.
  warn(


In [2]:
# Load test data (100 samples per class for quick verification)
_, _, test_rec = load_dataset(max_per_class=100)

# Build ONLY test loader
test_dataset = CropDiseaseDataset(test_rec, transform=get_eval_transforms())
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Test Loader ready with {len(test_dataset)} samples.")

# Load the trained model
# Note: The constructor uses CFG directly
model = HybridQuantumClassifier().to(CFG.DEVICE)
checkpoint_path = os.path.join(CFG.MODELS_DIR, 'hybrid_vqc_best.pt')
model.load_state_dict(torch.load(checkpoint_path, map_location=CFG.DEVICE))
model.eval()
print(f"Model loaded from: {checkpoint_path}")

Dataset loaded → train: 1540, val: 220, test: 440
  Train class distribution:
    [0] apple_apple_scab               → 71 images
    [1] apple_black_rot                → 74 images
    [2] apple_cedar_apple_rust         → 73 images
    [3] apple_healthy                  → 78 images
    [4] grape_black_rot                → 71 images
    [5] grape_esca_black_measles       → 71 images
    [6] grape_healthy                  → 68 images
    [7] grape_leaf_blight_isariopsis_leaf_spot → 61 images
    [8] potato_early_blight            → 68 images
    [9] potato_healthy                 → 67 images
    [10] potato_late_blight             → 64 images
    [11] tomato_bacterial_spot          → 66 images
    [12] tomato_early_blight            → 77 images
    [13] tomato_healthy                 → 70 images
    [14] tomato_late_blight             → 66 images
    [15] tomato_septoria_leaf_spot      → 82 images
    [16] tomato_spider_mites_two_spotted_spider_mite → 61 images
    [17] tomato_target_spot

c:\users\balaj\appdata\local\programs\python\python38\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\users\balaj\appdata\local\programs\python\python38\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model loaded from: C:\Files\PROJECTS\QuantumCrop\outputs\models\hybrid_vqc_best.pt


C:\Users\balaj\AppData\Local\Temp\ipykernel_7096\3464240078.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path, map_locati

In [3]:
# Run Evaluation
results = evaluate_model(model, test_loader, 'AgriLens 22-Class Hybrid VQC Live Verification')
print(f"\nFinal Verification Accuracy: {results['accuracy']:.4f}")


AgriLens 22-Class Hybrid VQC Live Verification Results:
  Accuracy  : 0.8682
  Precision : 0.8779
  Recall    : 0.8730
  Macro F1  : 0.8719

                                             precision    recall  f1-score   support

                           apple_apple_scab       0.89      0.73      0.80        22
                            apple_black_rot       0.94      0.94      0.94        17
                     apple_cedar_apple_rust       0.94      0.94      0.94        18
                              apple_healthy       0.84      0.94      0.89        17
                            grape_black_rot       1.00      0.90      0.95        21
                   grape_esca_black_measles       0.95      1.00      0.98        21
                              grape_healthy       0.95      1.00      0.97        19
     grape_leaf_blight_isariopsis_leaf_spot       1.00      1.00      1.00        22
                        potato_early_blight       1.00      0.89      0.94        18
       